# TP-3 — TinyGPT: export a ONNX y un inference server

Autor: Msc. Abraham R.

Vas a tomar el TinyGPT que fine-tuneaste en el TP-2 y ponerlo en producción: sacarlo de
PyTorch, meterlo en un grafo de cómputo portable, y ponerlo detrás de una API compatible
con OpenAI.

## De un framework de entrenamiento a un runtime

Acá no se entrena nada. El TP-1 preentrenó un modelo y el TP-2 le enseñó a seguir
instrucciones; los dos corrieron dentro de un notebook, en PyTorch, un request por vez.
Producción no funciona así, porque un framework de entrenamiento está hecho para ser
flexible y un runtime está hecho para una sola tarea:

| | PyTorch | un runtime de inferencia |
|---|---|---|
| Hecho para | experimentar, autograd | un grafo fijo, ejecutado rápido |
| El modelo es | código Python + pesos | un grafo serializado, sin Python |
| Se entrega como | un repo y un entorno | un solo archivo |
| Corre en | donde instale PyTorch | CPU, GPU, NPU, teléfono, browser |

Este es el paso que hay detrás de runtimes como:
- ONNX Runtime (el que vas a usar acá)
- TensorRT, OpenVINO, CoreML
- llama.cpp / GGUF, que recorrimos en clase

**Parte 1 — Export.** Convertí el modelo en un grafo ONNX y demostrá que el grafo calcula
lo que calculaba PyTorch. Después exportá un segundo grafo que lleve el KV-cache.

**Parte 2 — Cuantizar y medir.** Hacelo 4x más chico, y después averiguá honestamente qué
costó eso en calidad y qué compró en velocidad.

**Parte 3 — Servir.** Poné el grafo detrás de `POST /v1/chat/completions` y llamalo con el
cliente oficial de OpenAI.

El modelo no cambia en este notebook. Es el mismo checkpoint de punta a punta — lo único
que cambia es su representación.

## Consignas

**Parte 1**
- Consigna I — exportar a ONNX, y escribir el test de paridad que lo demuestra.
- Consigna II — exportar un segundo grafo que reciba y devuelva el **KV-cache**.
- Consigna III — portar tu `generateV2` del TP-1 al runtime: temperatura, top-k, top-p, y
  el cache.

**Parte 2**
- Consigna IV — cuantizar a **INT8**, y medir qué costó en comportamiento.
- Consigna V — benchmarkear PyTorch vs ONNX vs INT8, y separar el tiempo de engine del de
  Python.

**Parte 3**
- Consigna VI — implementar `POST /v1/chat/completions`, compatible con OpenAI.
- Consigna VII — agregar **streaming**: el mismo endpoint, token por token sobre SSE.
- Consigna VIII — llamar a tu server con el cliente oficial `openai`, de las dos formas.

## Cómo trabajar en esto

**Requisitos previos:** el TP-2. Este notebook carga
`./checkpoints/tp2_full_ft/checkpoint_final.pt`. Sin ese archivo todo sigue corriendo,
con pesos al azar — los números siguen teniendo sentido, el texto generado no.

Las Consignas I, II, III, IV y VI vienen con una celda `check_*()`. Corréla antes de
seguir. Los bugs de paridad son silenciosos: un grafo equivocado devuelve números
plausibles, y no te vas a enterar hasta que las generaciones estén sutilmente mal.

**Acá las versiones importan.** `torch.onnx.export` es una API en movimiento, y el
comportamiento del exportador cambia entre releases. Las versiones contra las que se
escribió este notebook las imprime la primera celda; si las tuyas son distintas y algo se
rompe, eso es lo primero que hay que mirar.

Referencias: [ONNX](https://onnx.ai/) ·
[ONNX Runtime](https://onnxruntime.ai/docs/) ·
[torch.onnx](https://pytorch.org/docs/stable/onnx.html) ·
[chat completions de OpenAI](https://platform.openai.com/docs/api-reference/chat)

In [1]:
import json
import os
import re
import threading
import time
import uuid
from collections import Counter
from typing import Dict, List, Literal, Optional, Sequence, Tuple

import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer

import onnx
import onnxruntime as ort
import requests
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

from tinygpt import (
    GPTConfig,
    TinyGPT,
    describe,
    load_checkpoint,
    load_tinyshakespeare,
)

torch.manual_seed(1337)

# Todo acá corre en CPU a propósito: es el target que este notebook puede medir
# honestamente, y el que todo el mundo tiene.
print(f"torch        {torch.__version__}")
print(f"onnx         {onnx.__version__}")
print(f"onnxruntime  {ort.__version__}")
print(f"providers    {ort.get_available_providers()}")

torch        2.14.0+cpu
onnx         1.23.1
onnxruntime  1.23.2
providers    ['AzureExecutionProvider', 'CPUExecutionProvider']


# Etapa 0 — el modelo que vas a poner en producción

El fine-tune del TP-2, cargado exactamente como el TP-2 lo dejó: mismo tokenizer, mismos
cuatro tokens especiales, mismo embedding y head sin atar.

In [2]:
USER, ASSISTANT, END, PAD = "<|user|>", "<|assistant|>", "<|end|>", "<|pad|>"
TP2_CKPT = "./checkpoints/tp2_full_ft/checkpoint_final.pt"
ONNX_DIR = "./onnx"
os.makedirs(ONNX_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"additional_special_tokens": [USER, ASSISTANT, END, PAD]})
END_ID = tokenizer.convert_tokens_to_ids(END)
# El andamiaje que una respuesta nunca debe contener -- Consigna VI.
SCAFFOLD_IDS = [tokenizer.convert_tokens_to_ids(t) for t in (USER, ASSISTANT, PAD)]

# tie_weights=False por la misma razón que en el TP-2: los checkpoints mantienen
# token_emb y head como dos matrices separadas.
config = GPTConfig(vocab_size=len(tokenizer), tie_weights=False)
model = TinyGPT(config)

if os.path.exists(TP2_CKPT):
    load_checkpoint(model, TP2_CKPT, map_location="cpu")
    print(f"pesos fine-tuneados cargados desde {TP2_CKPT}")
else:
    print("no se encontró el checkpoint del TP-2 -- se sigue con pesos al azar.")
    print("(todas las mediciones del notebook siguen funcionando; el texto va a ser ruido)")

model.eval()
describe(model, "TinyGPT (fine-tuneado en el TP-2)")
print(f"\nblock_size {config.block_size} | n_layer {config.n_layer} | "
      f"n_head {config.n_head} | vocab {config.vocab_size:,}")

pesos fine-tuneados cargados desde ./checkpoints/tp2_full_ft/checkpoint_final.pt
TinyGPT (fine-tuneado en el TP-2): 6,535,552 parameters | 6,535,552 trainable (100.00%)

block_size 32 | n_layer 2 | n_head 4 | vocab 50,261


## Contra qué medimos el grafo

Se necesitan tres cosas en todo el notebook, y las tres están dadas: texto reservado para
puntuar, un conjunto de prompts, y un loop de decodificación.

**La accuracy no es una de ellas.** El TP-2 midió `who_said` prácticamente en el nivel de
azar sobre 98 personajes, así que una columna de accuracy acá daría 0% para cada artefacto
y no te diría nada sobre lo que hizo la cuantización. En su lugar usamos lo que sí se
mueve:

- **cross-entropy** sobre Shakespeare reservado — el objetivo de preentrenamiento, el
  mismo número que el TP-2 usó para medir el olvido. Es una medida real de calidad y
  funciona por malo que sea el modelo en cualquier tarea puntual;
- **coincidencia** con el modelo fp32 — con qué frecuencia dos artefactos toman la misma
  decisión.

Calidad y fidelidad son preguntas distintas, y después de cuantizar necesitás las dos.

In [3]:
# La misma extracción que el TP-2, sobre el mismo corpus, con el mismo shuffle 1337 --
# así que estos son ejemplos sobre los que el fine-tune no entrenó.
_text = load_tinyshakespeare(None)
_speeches = re.findall(r"^([A-Z][A-Za-z' ]{1,30}):\n((?:.+\n)+)", _text, flags=re.M)
_counts = Counter(s for s, _ in _speeches)
_speeches = [(s, t) for s, t in _speeches if _counts[s] >= 20]
import random as _random
_random.Random(1337).shuffle(_speeches)
_held_out = _speeches[int(0.9 * len(_speeches)):]

BUDGET = config.block_size + 1


def _who_said(speaker: str, speech: str) -> Optional[Tuple[str, str]]:
    line = " ".join(speech.split())
    room = BUDGET - len(tokenizer.encode(f"{USER}who said: {ASSISTANT}")) \
                  - len(tokenizer.encode(speaker + END))
    if room < 5:
        return None
    body = tokenizer.decode(tokenizer.encode(line)[:room])
    return f"{USER}who said: {body}{ASSISTANT}", speaker


EVAL_SET = [p for p in (_who_said(s, t) for s, t in _held_out) if p is not None]

# Texto reservado para la cross-entropy: el último 10% del corpus, en ventanas de block_size.
_ids = tokenizer(_text[-120_000:], add_special_tokens=False)["input_ids"]
LM_WINDOWS = [(np.array([_ids[i:i + config.block_size]], dtype=np.int64),
               np.array(_ids[i + 1:i + config.block_size + 1], dtype=np.int64))
              for i in range(0, 64 * config.block_size, config.block_size)]

# Algunos prompts para generar, para leer más que para puntuar.
DEMO_PROMPTS = [f"{USER}continue: To be, or not to be,{ASSISTANT}",
                f"{USER}who said: Now is the winter of our discontent{ASSISTANT}",
                f"{USER}reverse: sword{ASSISTANT}"]


def make_session(path: str, providers=("CPUExecutionProvider",)) -> ort.InferenceSession:
    """Una sesión de ONNX Runtime con la charlatanería de carga bajada."""
    so = ort.SessionOptions()
    so.log_severity_level = 3
    return ort.InferenceSession(path, so, providers=list(providers))


# Cada llamada a una sesión pasa por acá, así la Consigna V puede preguntar cuánto del
# reloj fue el engine y cuánto fue Python. Usalo en vez de sess.run.
ENGINE = {"seconds": 0.0}


def run_session(sess: ort.InferenceSession, feed: Dict[str, np.ndarray]) -> List[np.ndarray]:
    t0 = time.perf_counter()
    out = sess.run(None, feed)
    ENGINE["seconds"] += time.perf_counter() - t0
    return out


def _shape(value) -> str:
    t = value.type.tensor_type
    dims = [(d.dim_param or str(d.dim_value)) for d in t.shape.dim]
    return f"{onnx.TensorProto.DataType.Name(t.elem_type).lower()}[{','.join(dims)}]"


def show_graph(path: str, max_nodes: int = 12) -> None:
    """Firma, mezcla de operadores y los primeros nodos de un archivo ONNX."""
    g = onnx.load(path).graph
    stored = {i.name for i in g.initializer}
    size = sum(len(i.raw_data) for i in g.initializer) / 1e6
    print(f"{path}  |  {len(g.node)} nodos  |  {len(g.initializer)} tensores de pesos "
          f"({size:.1f} MB)")
    print("  in   " + ", ".join(f"{i.name}: {_shape(i)}" for i in g.input))
    print("  out  " + ", ".join(f"{o.name}: {_shape(o)}" for o in g.output))
    print(f"  ops  {dict(Counter(n.op_type for n in g.node).most_common(8))}")
    for n in g.node[:max_nodes]:
        args = ", ".join(("*" + a if a in stored else a) for a in n.input)
        print(f"    {n.output[0][:28]:<30} = {n.op_type}({args[:52]})")
    if len(g.node) > max_nodes:
        print(f"    ... {len(g.node) - max_nodes} más    (* = un peso guardado en el archivo)")


def greedy_torch(m: nn.Module, prompt: str, max_new_tokens: int = 14) -> str:
    """Loop de decodificación de referencia, sin cache, para comparar peras con peras."""
    ids = tokenizer.encode(prompt)
    produced = []
    for _ in range(max_new_tokens):
        x = torch.tensor(ids[-config.block_size:], dtype=torch.long)[None, :]
        with torch.no_grad():
            nxt = int(m(x)[0, -1].argmax())
        if nxt == END_ID:
            break
        produced.append(nxt)
        ids.append(nxt)
    return tokenizer.decode(produced)


def greedy_onnx(sess: ort.InferenceSession, prompt: str, max_new_tokens: int = 14) -> str:
    """El mismo loop contra una sesión cuya única entrada es `input_ids`."""
    ids = tokenizer.encode(prompt)
    produced = []
    for _ in range(max_new_tokens):
        x = np.array([ids[-config.block_size:]], dtype=np.int64)
        nxt = int(run_session(sess, {"input_ids": x})[0][0, -1].argmax())
        if nxt == END_ID:
            break
        produced.append(nxt)
        ids.append(nxt)
    return tokenizer.decode(produced)


def cross_entropy(next_logits) -> float:
    """
    Cross-entropy media sobre LM_WINDOWS, en nats por token.

    `next_logits(x) -> (T, vocab)` para una ventana `x` de forma (1, T).
    """
    total = 0.0
    for x, y in LM_WINDOWS:
        logits = next_logits(x).astype(np.float64)
        shift = logits - logits.max(axis=-1, keepdims=True)
        logZ = np.log(np.exp(shift).sum(axis=-1))
        total += float((logZ - shift[np.arange(len(y)), y]).mean())
    return total / len(LM_WINDOWS)


def torch_logits(x: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return model(torch.from_numpy(x))[0].numpy()


def onnx_logits(sess: ort.InferenceSession):
    return lambda x: sess.run(None, {"input_ids": x})[0][0]


print(f"{len(EVAL_SET):,} prompts who_said reservados | "
      f"{len(LM_WINDOWS)} ventanas de texto reservado")
print(f"cross-entropy de torch: {cross_entropy(torch_logits):.4f} nats/token")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (38796 > 1024). Running this sequence through the model will result in indexing errors


597 prompts who_said reservados | 64 ventanas de texto reservado
cross-entropy de torch: 5.6950 nats/token


# Consigna I — exportar a ONNX

Un archivo ONNX es un **grafo de cómputo**: los operadores, su orden y los pesos, sin nada
de Python. `torch.onnx.export` lo produce *trazando* — corriendo tu modelo una vez con una
entrada de mentira y registrando las operaciones que efectivamente se ejecutaron.

El trazado es la razón por la que un export puede estar mal sin que nada falle. Cualquier
cosa decidida por un `if` de Python sobre el valor de un tensor queda congelada como lo que
haya pasado en esa única corrida.

**Qué implementar**

1. Un `nn.Module` que envuelva al modelo. `TinyGPT.forward` devuelve una tupla cuando
   `use_cache` o `return_weights` están activados, y un tensor pelado si no. El tracer
   necesita exactamente un camino con una sola salida, así que envolvelo y devolvé solo
   `logits`.
2. `torch.onnx.export(...)` con:
   - `input_names=["input_ids"]`, `output_names=["logits"]`;
   - `dynamic_axes` marcando **batch** y **secuencia** como dinámicos en los dos. Sin eso
     el grafo queda congelado en la forma de la entrada de mentira y solo acepta prompts
     de ese largo exacto;
   - `opset_version=17`, `do_constant_folding=True`.

**Dos trampas, las dos silenciosas**

- **`export()` deja el módulo en modo entrenamiento.** El dropout queda vivo del lado de
  PyTorch, tu chequeo de paridad marca ~4 en vez de ~1e-06, y te pasás la tarde culpando al
  exportador. Llamá a `.eval()` de nuevo después de exportar — el self-check lo verifica.
- **La paridad es la única prueba que tenés.** Un grafo trazado que corre no es un grafo
  correcto. Compará los logits contra PyTorch en *varios* largos de secuencia, no en uno.

In [4]:
class ExportWrapper(nn.Module):
    """Una entrada, una salida, un camino por el modelo -- lo que el tracer necesita."""

    def __init__(self, m: nn.Module):
        super().__init__()
        # TODO: Consigna I
        self.m = m

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # TODO: Consigna I -- devolver solo los logits
        return self.m(input_ids)


def export_onnx(m: nn.Module, path: str) -> str:
    """
    Trazá `m` a un grafo ONNX en `path`, con batch y secuencia dinámicos.

    Devuelve `path`. Acordate de lo que el trazado le hace a `m.training`.
    """
    # TODO: Consigna I
    wrapper = ExportWrapper(m).eval()
    dummy = torch.randint(0, config.vocab_size, (1, min(8, config.block_size)),
                          dtype=torch.long)

    torch.onnx.export(
        wrapper,
        dummy,
        path,
        input_names=["input_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "sequence"},
            "logits": {0: "batch", 1: "sequence"},
        },
        opset_version=17,
        dynamo=False,
        do_constant_folding=True,
    )
    m.eval()
    return path

## Self-check

In [5]:
def check_export():
    path = export_onnx(model, f"{ONNX_DIR}/tinygpt.onnx")
    assert os.path.exists(path), "no se escribió nada"

    assert not model.training, \
        "el modelo está en modo entrenamiento -- lo dejó export(); llamá a .eval()"

    sess = make_session(path)
    names = [i.name for i in sess.get_inputs()]
    assert names == ["input_ids"], f"las entradas son {names}"

    worst = 0.0
    for T in (1, 8, 17, config.block_size):
        x = torch.randint(0, config.vocab_size, (1, T), dtype=torch.long)
        with torch.no_grad():
            ref = model(x).numpy()
        got = sess.run(None, {"input_ids": x.numpy()})[0]
        assert got.shape == ref.shape, f"T={T}: {got.shape} != {ref.shape}"
        worst = max(worst, float(np.abs(ref - got).max()))
    assert worst < 1e-4, f"los logits difieren en {worst:.2e} -- el grafo no es el modelo"

    # Un eje de batch dinámico es lo que después le permite a un server batchear requests.
    x = torch.randint(0, config.vocab_size, (4, 8), dtype=torch.long)
    assert sess.run(None, {"input_ids": x.numpy()})[0].shape[0] == 4, \
        "el eje de batch quedó congelado -- revisá dynamic_axes"

    print(f"paridad    max |torch - onnx| = {worst:.2e}  (seq 1/8/17/{config.block_size})")
    print(f"muestra    {greedy_onnx(sess, DEMO_PROMPTS[1])!r}")
    print("todos los checks pasaron\n")
    # El mismo lector de recién -- ahora apuntado a un transformer.
    show_graph(path, max_nodes=6)
    return sess


fp32_sess = check_export()

C:\Users\Admin\AppData\Local\Temp\ipykernel_8176\2272730733.py:25: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
c:\Users\Admin\MIA\PLN2\NLP2\TP-III\tinygpt.py:311: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert past_len + T <= self.config.block_size, (


paridad    max |torch - onnx| = 1.72e-05  (seq 1/8/17/32)
muestra    'QUEEN ELIZABETH'
todos los checks pasaron

./onnx/tinygpt.onnx  |  221 nodos  |  30 tensores de pesos (26.1 MB)
  in   input_ids: int64[batch,sequence]
  out  logits: float[batch,sequence,50261]
  ops  {'Constant': 78, 'Unsqueeze': 25, 'Add': 15, 'Gather': 13, 'MatMul': 13, 'Shape': 11, 'Slice': 10, 'Transpose': 10}
    m.blocks.1.attn.tril           = Identity(*m.blocks.0.attn.tril)
    /m/Shape_output_0              = Shape(input_ids)
    /m/Constant_output_0           = Constant()
    /m/Gather_output_0             = Gather(/m/Shape_output_0, /m/Constant_output_0)
    /m/token_emb/Gather_output_0   = Gather(*m.token_emb.weight, input_ids)
    /m/Constant_1_output_0         = Constant()
    ... 215 más    (* = un peso guardado en el archivo)


## Mirá lo que acabás de fabricar

`check_export` imprimió la firma del grafo y su mezcla de operadores. Ahora abrí el archivo.

[**Netron**](https://netron.app) es el visor estándar para este formato, y la celda de abajo
lo embebe en el notebook si lo tenés instalado (`pip install netron`). Si no, abrí
[netron.app](https://netron.app) y arrastrá `onnx/tinygpt.onnx` sobre la página — se
renderiza localmente en tu browser, el archivo no se sube a ningún lado.

Cosas que vale la pena encontrar, porque son el punto del ejercicio:

- **Un `Gather` al principio y un `MatMul` de 50.261 de ancho al final** — el embedding y
  la cabeza de salida, las dos matrices que el TP-2 midió como el 98% de este modelo.
- **Tu máscara de atención de `block_size`, horneada como constante.** Es un `Slice` sobre
  un `tril` guardado, no código.
- **Nada de Python, ni clases, ni `forward`.** `TinyGPT` desapareció; lo que queda es
  aritmética.
- **Dos bloques que se ven idénticos.** Tu `n_layer = 2`, desenrollado — el loop que los
  escribió no existe en el archivo.

Encontrá un operador que vos no escribiste y explicá de dónde salió. El trazado agrega
nodos `Shape`, `Unsqueeze` y `Constant` por cada cuenta de formas que tu `forward` hizo a
nivel de Python, que es la razón por la que salen 221 nodos de un modelo que podés leer en
un minuto.

In [6]:
def view_graph(path: str, height: int = 600):
    """Embebe Netron si está instalado; si no, dice cómo ver el archivo a mano."""
    try:
        import netron
    except ImportError:
        print(f"netron no está instalado. O bien:\n"
              f"  pip install netron      (y volvé a correr esta celda)\n"
              f"  o abrí https://netron.app y arrastrá {path}")
        return show_graph(path, max_nodes=12)

    from IPython.display import IFrame
    host, port = netron.start(path, address=("localhost", 0), browse=False)
    print(f"sirviendo {path} en http://{host}:{port}   (netron.stop() para cerrarlo)")
    return IFrame(f"http://{host}:{port}", width="100%", height=height)


view_graph(f"{ONNX_DIR}/tinygpt.onnx")

netron no está instalado. O bien:
  pip install netron      (y volvé a correr esta celda)
  o abrí https://netron.app y arrastrá ./onnx/tinygpt.onnx
./onnx/tinygpt.onnx  |  221 nodos  |  30 tensores de pesos (26.1 MB)
  in   input_ids: int64[batch,sequence]
  out  logits: float[batch,sequence,50261]
  ops  {'Constant': 78, 'Unsqueeze': 25, 'Add': 15, 'Gather': 13, 'MatMul': 13, 'Shape': 11, 'Slice': 10, 'Transpose': 10}
    m.blocks.1.attn.tril           = Identity(*m.blocks.0.attn.tril)
    /m/Shape_output_0              = Shape(input_ids)
    /m/Constant_output_0           = Constant()
    /m/Gather_output_0             = Gather(/m/Shape_output_0, /m/Constant_output_0)
    /m/token_emb/Gather_output_0   = Gather(*m.token_emb.weight, input_ids)
    /m/Constant_1_output_0         = Constant()
    /m/Cast_output_0               = Cast(/m/Gather_output_0)
    /m/Constant_2_output_0         = Constant()
    /m/Range_output_0              = Range(/m/Constant_1_output_0, /m/Cast_output_0, /

# Consigna II — exportar el KV-cache

El grafo que acabás de exportar vuelve a leer el prompt entero para cada token. La Consigna
III del TP-1 midió lo que ahorra un KV-cache; ahora tiene que sobrevivir el viaje a ONNX.

Un runtime no tiene objetos de Python donde guardar estado entre llamadas, así que el cache
no puede vivir en una lista al costado. **Pasa a ser entrada y salida del grafo**: quien
llama pasa el pasado y recibe el presente. Así exactamente se sirven GPT-2, Llama y
cualquier otro decoder exportado.

`TinyGPT` guarda un tensor apilado por capa, de forma
`(2, batch, n_head, past, head_dim)` — key y value apilados en el eje 0. Con `n_layer = 2`
eso son 2 tensores de entrada y 2 de salida, más `input_ids`.

**Qué implementar**

1. `KVWrapper`, cuyo `forward(input_ids, *past)` llama al modelo con
   `kv_cache=list(past), use_cache=True` y devuelve `(logits, *present)`.
2. `export_onnx_kv`, trazándolo con un cache real de un forward real, nombrando los
   tensores `past_0..` y `present_0..`, y marcando el **eje del largo del pasado como
   dinámico** en todos ellos.

**Acá la trampa es la máscara.** La atención corta `tril[past:T_k, :T_k]`, donde `past`
sale de la forma de un tensor. Si ese corte se traza como constante, el grafo funciona
perfecto en el largo con el que lo trazaste y está mal en todos los demás — por eso el
self-check corre varios largos de pasado, no uno.

**Pista.** Armá el cache con el que vas a trazar así:

```python
with torch.no_grad():
    _, cache = m(some_ids, use_cache=True)
```

In [7]:
class KVWrapper(nn.Module):
    """Cache que entra, cache que sale -- un runtime no guarda estado entre llamadas."""

    def __init__(self, m: nn.Module):
        super().__init__()
        # TODO: Consigna II
        self.m = m

    def forward(self, input_ids: torch.Tensor, *past: torch.Tensor):
        # TODO: Consigna II -- devolver (logits, *present)
        logits, present = self.m(input_ids, kv_cache=list(past), use_cache=True)
        return (logits, *present)


def export_onnx_kv(m: nn.Module, path: str) -> str:
    """
    Exportá un grafo de un paso que recibe el cache pasado y devuelve el presente.

    Entradas: input_ids, past_0 .. past_{n_layer-1}
    Salidas:  logits,    present_0 .. present_{n_layer-1}
    """
    # TODO: Consigna II
    wrapper = KVWrapper(m).eval()

    # Trazamos con un cache real, no con tensores inventados: así las formas y dtypes
    # coinciden exactamente con los que produce TinyGPT.
    some_ids = torch.randint(0, config.vocab_size, (1, min(4, config.block_size - 1)),
                             dtype=torch.long)
    with torch.no_grad():
        _, cache = m(some_ids, use_cache=True)

    step = torch.randint(0, config.vocab_size, (1, 1), dtype=torch.long)
    args = (step, *cache)
    input_names = ["input_ids"] + [f"past_{i}" for i in range(config.n_layer)]
    output_names = ["logits"] + [f"present_{i}" for i in range(config.n_layer)]

    dynamic_axes = {
        "input_ids": {0: "batch", 1: "sequence"},
        "logits": {0: "batch", 1: "sequence"},
    }
    for i in range(config.n_layer):
        dynamic_axes[f"past_{i}"] = {1: "batch", 3: "past_sequence"}
        dynamic_axes[f"present_{i}"] = {1: "batch", 3: "present_sequence"}

    torch.onnx.export(
        wrapper,
        args,
        path,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        opset_version=17,
        dynamo=False,
        do_constant_folding=True,
    )
    m.eval()
    return path

## Self-check

In [8]:
def empty_cache(batch: int = 1) -> List[np.ndarray]:
    """Lo que alimentás en el primer paso: un cache con cero posiciones de pasado."""
    head_dim = config.n_embd // config.n_head
    return [np.zeros((2, batch, config.n_head, 0, head_dim), dtype=np.float32)
            for _ in range(config.n_layer)]


def check_kv_export():
    path = export_onnx_kv(model, f"{ONNX_DIR}/tinygpt.kv.onnx")
    assert not model.training, "export() volvió a dejar el modelo en modo entrenamiento"

    sess = make_session(path)
    ins = [i.name for i in sess.get_inputs()]
    outs = [o.name for o in sess.get_outputs()]
    assert ins == ["input_ids"] + [f"past_{i}" for i in range(config.n_layer)], ins
    assert outs == ["logits"] + [f"present_{i}" for i in range(config.n_layer)], outs

    # El punto del test: largos de pasado que el export nunca vio.
    worst = 0.0
    for past_len in (0, 3, 8, 20, config.block_size - 1):
        ids = torch.randint(0, config.vocab_size, (1, max(past_len, 1)), dtype=torch.long)
        if past_len == 0:
            cache = empty_cache()
        else:
            with torch.no_grad():
                _, c = model(ids, use_cache=True)
            cache = [t.numpy() for t in c]

        step = torch.randint(0, config.vocab_size, (1, 1), dtype=torch.long)
        feed = {"input_ids": step.numpy()}
        feed.update({f"past_{i}": c for i, c in enumerate(cache)})
        got = sess.run(None, feed)

        torch_cache = None if past_len == 0 else [torch.from_numpy(c) for c in cache]
        with torch.no_grad():
            ref, present = model(step, kv_cache=torch_cache, use_cache=True)
        worst = max(worst, float(np.abs(ref.numpy() - got[0]).max()))
        assert got[1].shape[3] == past_len + 1, \
            f"el cache presente mide {got[1].shape[3]}, se esperaba {past_len + 1}"
        assert np.abs(present[0].numpy() - got[1]).max() < 1e-4, "el cache devuelto difiere"

    assert worst < 1e-4, (f"los logits difieren en {worst:.2e} -- lo más probable es que la "
                          f"máscara se haya trazado como constante")
    print(f"paridad    max |torch - onnx| = {worst:.2e}  (past 0/3/8/20/{config.block_size - 1})")
    print("todos los checks pasaron\n")
    # Leé la firma: ahora el cache es parte del contrato, de entrada y de salida.
    show_graph(path, max_nodes=0)
    return sess


kv_sess = check_kv_export()

C:\Users\Admin\AppData\Local\Temp\ipykernel_8176\880667139.py:45: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


paridad    max |torch - onnx| = 1.05e-05  (past 0/3/8/20/31)
todos los checks pasaron

./onnx/tinygpt.kv.onnx  |  251 nodos  |  30 tensores de pesos (26.1 MB)
  in   input_ids: int64[batch,sequence], past_0: float[2,batch,4,past_sequence,16], past_1: float[2,batch,4,past_sequence,16]
  out  logits: float[batch,sequence,50261], present_0: float[2,batch,4,present_sequence,16], present_1: float[2,batch,4,present_sequence,16]
  ops  {'Constant': 88, 'Unsqueeze': 29, 'Add': 16, 'Gather': 14, 'Concat': 14, 'MatMul': 13, 'Shape': 12, 'Slice': 10}
    ... 251 más    (* = un peso guardado en el archivo)


# Consigna III — `generateV2`, portado al runtime

La Consigna I del TP-1 te hizo escribir `generateV2`: greedy, temperatura, top-k, top-p,
con un KV-cache opcional. Eso era PyTorch. **La decodificación no es parte del modelo** —
nunca lo fue, y por eso nada de eso aparece en el grafo que exportaste. Es un loop que
escribe quien llama, alrededor de lo que sea que calcule los logits.

Así que portalo. Las mismas perillas, la misma semántica que en el TP-1, contra una sesión
de ONNX en vez de un `nn.Module`. Dos diferencias, y son la lección:

- **No hay `model.forward`, solo un diccionario de entradas.** El truncado, el manejo del
  cache y la contabilidad de posiciones ahora son tuyos.
- **El cache no es estado escondido.** Con `use_cache=True` manejás el grafo de la Consigna
  II: alimentás `past_*`, leés `present_*`, y los devolvés en el paso siguiente.

**Qué implementar**

1. `sample_next(logits, ...)` — las reglas de decodificación, como función pura sobre una
   fila de logits. Greedy cuando `do_sample=False`; si no, temperatura, después top-k,
   después top-p, después samplear. `suppress` lista ids de tokens que nunca pueden
   elegirse (la Parte 3 lo necesita para el andamiaje del chat; la Consigna V del TP-2
   chocó con el mismo problema).
2. `stream_tokens(sess, prompt, ...)` — un **generador** que va entregando ids de tokens
   hasta `<|end|>` o `max_new_tokens`. `use_cache=False` maneja el grafo de la Consigna I,
   `use_cache=True` el de la Consigna II.

Un generador y no un string, porque la Parte 3 necesita los tokens a medida que aparecen.
`generate_onnx`, abajo, lo vuelve a envolver en texto y está dado.

**Detalles que importan**

- Ordená las perillas como en el TP-1: la temperatura escala los logits, después top-k
  trunca, después top-p trunca, después sampleás. Aplicar top-p antes de la temperatura da
  otra distribución.
- `top_p` se queda con el **conjunto más chico cuya probabilidad acumulada llega a p**, así
  que `top_p` cerca de 0 es prácticamente greedy, y `top_p=1.0` no trunca nada.
- Con el cache, después de la primera llamada solo entra el token **nuevo** como
  `input_ids`, y el cache hay que recortarlo en `block_size` — el grafo lo verifica.
- Llamá a `run_session(sess, feed)` en vez de `sess.run(...)`, para que la Consigna V pueda
  medir cuánto del tiempo fue realmente el engine.
- Sembrá con `np.random.default_rng(seed)` para poder reproducir una corrida sampleada.

In [9]:
def sample_next(logits: np.ndarray, do_sample: bool = True,
                temperature: float = 1.0, top_k: Optional[int] = None,
                top_p: Optional[float] = None, suppress: Sequence[int] = (),
                rng: Optional[np.random.Generator] = None) -> int:
    """
    Una fila de logits -> un id de token.

    do_sample=False -> arg-max, y todas las otras perillas se ignoran (como en el TP-1).
    Orden: temperatura, después top-k, después top-p, después samplear.
    """
    # TODO: Consigna III
    scores = np.asarray(logits, dtype=np.float64).reshape(-1).copy()

    if suppress:
        valid = [int(i) for i in suppress if 0 <= int(i) < scores.size]
        scores[valid] = -np.inf

    if not np.isfinite(scores).any():
        raise ValueError("todos los tokens quedaron suprimidos")

    if not do_sample:
        return int(np.argmax(scores))

    if temperature <= 0:
        raise ValueError("temperature debe ser > 0 cuando do_sample=True")
    scores /= float(temperature)

    if top_k is not None:
        k = max(1, min(int(top_k), scores.size))
        if k < scores.size:
            idx = np.argpartition(scores, -k)[-k:]
            keep = np.zeros(scores.size, dtype=bool)
            keep[idx] = True
            scores[~keep] = -np.inf

    if top_p is not None and float(top_p) < 1.0:
        p = max(float(top_p), 0.0)
        order = np.argsort(scores)[::-1]
        ordered = scores[order]
        finite = np.isfinite(ordered)
        probs = np.zeros_like(ordered, dtype=np.float64)
        if finite.any():
            z = ordered[finite] - np.max(ordered[finite])
            ez = np.exp(z)
            probs[finite] = ez / ez.sum()
        cumulative = np.cumsum(probs)
        cutoff = int(np.searchsorted(cumulative, p, side="left"))
        cutoff = min(cutoff, len(order) - 1)
        keep_idx = order[:cutoff + 1]
        keep = np.zeros(scores.size, dtype=bool)
        keep[keep_idx] = True
        scores[~keep] = -np.inf

    finite = np.isfinite(scores)
    shifted = scores[finite] - np.max(scores[finite])
    probs_finite = np.exp(shifted)
    probs_finite /= probs_finite.sum()
    probs = np.zeros(scores.size, dtype=np.float64)
    probs[finite] = probs_finite

    if rng is None:
        rng = np.random.default_rng()
    return int(rng.choice(scores.size, p=probs))


def stream_tokens(sess: ort.InferenceSession, prompt: str, max_new_tokens: int = 24,
                  use_cache: bool = True, do_sample: bool = True,
                  temperature: float = 1.0, top_k: Optional[int] = None,
                  top_p: Optional[float] = None, suppress: Sequence[int] = (),
                  seed: Optional[int] = None):
    """
    Va entregando ids de tokens generados, hasta END_ID o max_new_tokens.

    use_cache=False -> `sess` es el grafo de la Consigna I (solo input_ids).
    use_cache=True  -> `sess` es el de la Consigna II (past_* adentro, present_* afuera).
    """
    # TODO: Consigna III
    ids = list(tokenizer.encode(prompt))
    rng = np.random.default_rng(seed)

    if use_cache:
        cache = empty_cache(batch=1)
        first = True

        for _ in range(max_new_tokens):
            if first:
                x = np.array([ids[-config.block_size:]], dtype=np.int64)
                first = False
            else:
                x = np.array([[ids[-1]]], dtype=np.int64)
                max_past = max(config.block_size - x.shape[1], 0)
                cache = [c[:, :, :, -max_past:, :] if max_past > 0
                         else c[:, :, :, :0, :] for c in cache]

            feed = {"input_ids": x}
            feed.update({f"past_{i}": c for i, c in enumerate(cache)})
            out = run_session(sess, feed)
            logits, cache = out[0], list(out[1:])

            nxt = sample_next(
                logits[0, -1],
                do_sample=do_sample,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                suppress=suppress,
                rng=rng,
            )
            if nxt == END_ID:
                break
            yield nxt
            ids.append(nxt)
    else:
        for _ in range(max_new_tokens):
            x = np.array([ids[-config.block_size:]], dtype=np.int64)
            logits = run_session(sess, {"input_ids": x})[0]
            nxt = sample_next(
                logits[0, -1],
                do_sample=do_sample,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                suppress=suppress,
                rng=rng,
            )
            if nxt == END_ID:
                break
            yield nxt
            ids.append(nxt)


def generate_onnx(sess: ort.InferenceSession, prompt: str, **kw) -> str:
    """Dado: el wrapper que devuelve texto, para que todos coincidan en el contrato."""
    return tokenizer.decode(list(stream_tokens(sess, prompt, **kw)))

## Self-check

In [10]:
def check_generate():
    prompt = DEMO_PROMPTS[1]
    greedy = dict(do_sample=False, max_new_tokens=14)

    # 1. el camino sin cache tiene que reproducir exactamente el loop de referencia
    mine = generate_onnx(fp32_sess, prompt, use_cache=False, **greedy)
    assert mine == greedy_onnx(fp32_sess, prompt), f"{mine!r} != referencia"

    # 2. y el camino con cache tiene que coincidir -- mismos pesos, mismas decisiones
    cached = generate_onnx(kv_sess, prompt, use_cache=True, **greedy)
    assert cached == mine, f"el cache cambia la salida: {cached!r} != {mine!r}"

    # 3. do_sample=False ignora todas las demás perillas
    loud = generate_onnx(kv_sess, prompt, temperature=5.0, top_k=999, top_p=0.3, **greedy)
    assert loud == mine, "do_sample=False tiene que ignorar temperature/top_k/top_p"

    # 4. los valores degenerados de cada perilla colapsan a greedy
    assert generate_onnx(kv_sess, prompt, max_new_tokens=14, top_k=1, seed=0) == mine, \
        "top_k=1 tiene que ser greedy"
    assert generate_onnx(kv_sess, prompt, max_new_tokens=14, top_p=1e-6, seed=0) == mine, \
        "top_p -> 0 se queda solo con el token más probable"

    # 5. el sampleo realmente samplea, y una semilla lo hace reproducible
    draws = {generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=s)
             for s in range(8)}
    assert len(draws) > 1, "temperature=2.0 dio una sola salida -- no está sampleando"
    a = generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=7)
    b = generate_onnx(kv_sess, prompt, max_new_tokens=14, temperature=2.0, seed=7)
    assert a == b, "la misma semilla tiene que dar el mismo texto"

    # 6. los ids suprimidos no se emiten nunca, por más caliente que sea el sampleo
    banned = [int(np.array(tokenizer.encode(mine))[0])] if mine else []
    if banned:
        for seed in range(4):
            ids = list(stream_tokens(kv_sess, prompt, max_new_tokens=14, temperature=2.0,
                                     suppress=banned, seed=seed))
            assert banned[0] not in ids, "se sampleó un id suprimido"

    # 7. sample_next es una función pura -- probala sin modelo ninguno
    logits = np.array([0.0, 10.0, 5.0, -1.0])
    assert sample_next(logits, do_sample=False) == 1
    assert sample_next(logits, do_sample=False, suppress=[1]) == 2
    rng = np.random.default_rng(0)
    assert sample_next(logits, top_k=1, rng=rng) == 1

    print(f"greedy       {mine!r}")
    print(f"con cache    {cached!r}  (idéntico)")
    print(f"temp 2.0     {sorted(draws)[0]!r}")
    print(f"salidas distintas con 8 semillas: {len(draws)}")
    print("todos los checks pasaron")


check_generate()

greedy       'QUEEN ELIZABETH'
con cache    'QUEEN ELIZABETH'  (idéntico)
temp 2.0     ' flower WhatsApperbON circum withhold, tx barn hatestr mean feel Once'
salidas distintas con 8 semillas: 8
todos los checks pasaron


# Consigna IV — INT8, y lo que cuesta

La cuantización dinámica guarda los pesos como enteros de 8 bits y los descuantiza al
vuelo. No necesita datos de calibración y es una sola llamada:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QUInt8)
```

**El test de la Consigna I ya no aplica.** La cuantización cambia la aritmética a
propósito. La paridad de logits contra PyTorch va a ser grande — del orden de 1e-01 en vez
de 1e-06 — y eso no es un bug para arreglar. Significa que cambió la pregunta: ya no es
*¿es el mismo cálculo?* sino **¿es el mismo modelo, en comportamiento?**

Así que medí comportamiento, en dos ejes:

- **calidad** — cross-entropy sobre texto reservado, fp32 contra INT8. ¿El modelo empeoró
  modelando lenguaje?
- **fidelidad** — coincidencia con fp32. La **coincidencia top-1** es con qué frecuencia
  los dos eligen el mismo token siguiente; la **coincidencia de respuestas** es con qué
  frecuencia un decode greedy entero sale idéntico carácter por carácter. La segunda es
  mucho más dura, porque un solo token distinto al principio cambia todo lo que sigue.

Reportá juntos tamaño, paridad, cross-entropy y las dos coincidencias. Uno de esos números
va a parecer alarmante y el resto no; tu informe tiene que decir cuál creés y por qué.

El self-check de abajo verifica la *medición*, no el modelo: una comparación que no puede
reconocer una sesión como idéntica a sí misma volvería ruido todos estos números.

In [11]:
def quantize_int8(fp32_path: str, int8_path: str) -> str:
    """Cuantización dinámica a INT8 de un archivo ONNX existente. Devuelve int8_path."""
    # TODO: Consigna IV
    from onnxruntime.quantization import QuantType, quantize_dynamic
    quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QUInt8)
    return int8_path


def top1_agreement(sess_a: ort.InferenceSession, sess_b: ort.InferenceSession,
                   n: int = 60) -> float:
    """
    Proporción de prompts donde las dos sesiones predicen el mismo token siguiente.

    Las dos sesiones toman solo `input_ids`. Un forward cada una, sin decodificar.
    """
    # TODO: Consigna IV
    prompts = EVAL_SET[:min(n, len(EVAL_SET))]
    same = 0
    for prompt, _ in prompts:
        ids = tokenizer.encode(prompt)[-config.block_size:]
        x = np.array([ids], dtype=np.int64)
        a = int(sess_a.run(None, {"input_ids": x})[0][0, -1].argmax())
        b = int(sess_b.run(None, {"input_ids": x})[0][0, -1].argmax())
        same += (a == b)
    return same / len(prompts) if prompts else 1.0


def reply_agreement(sess_a: ort.InferenceSession, sess_b: ort.InferenceSession,
                    n: int = 40) -> float:
    """Proporción de prompts donde los dos decodes greedy son strings idénticos."""
    # TODO: Consigna IV
    prompts = EVAL_SET[:min(n, len(EVAL_SET))]
    same = 0
    for prompt, _ in prompts:
        a = greedy_onnx(sess_a, prompt, max_new_tokens=24)
        b = greedy_onnx(sess_b, prompt, max_new_tokens=24)
        same += (a == b)
    return same / len(prompts) if prompts else 1.0


int8_path = quantize_int8(f"{ONNX_DIR}/tinygpt.onnx", f"{ONNX_DIR}/tinygpt.int8.onnx")
int8_sess = make_session(int8_path)

# TODO: Consigna IV -- completá esta tabla
#   tamaño MB | max |torch - onnx| | cross-entropy | coincidencia top-1 | de respuestas
def _max_parity(sess: ort.InferenceSession) -> float:
    worst = 0.0
    for T in (1, 8, 17, config.block_size):
        x = torch.randint(0, config.vocab_size, (1, T), dtype=torch.long)
        with torch.no_grad():
            ref = model(x).numpy()
        got = sess.run(None, {"input_ids": x.numpy()})[0]
        worst = max(worst, float(np.abs(ref - got).max()))
    return worst


fp32_metrics = {
    "size_mb": os.path.getsize(f"{ONNX_DIR}/tinygpt.onnx") / 1e6,
    "parity": _max_parity(fp32_sess),
    "cross_entropy": cross_entropy(onnx_logits(fp32_sess)),
    "top1": 1.0,
    "replies": 1.0,
}
int8_metrics = {
    "size_mb": os.path.getsize(int8_path) / 1e6,
    "parity": _max_parity(int8_sess),
    "cross_entropy": cross_entropy(onnx_logits(int8_sess)),
    "top1": top1_agreement(fp32_sess, int8_sess),
    "replies": reply_agreement(fp32_sess, int8_sess),
}
QUANT_METRICS = {"fp32": fp32_metrics, "int8": int8_metrics}

print(f"{'modelo':<8} {'MB':>8} {'max |torch-onnx|':>18} {'xent':>10} "
      f"{'top-1':>10} {'respuestas':>12}")
for name, m in QUANT_METRICS.items():
    print(f"{name:<8} {m['size_mb']:>8.2f} {m['parity']:>18.3e} "
          f"{m['cross_entropy']:>10.4f} {m['top1']:>9.1%} {m['replies']:>11.1%}")

print(f"\nINT8 ocupa {int8_metrics['size_mb'] / fp32_metrics['size_mb']:.2%} del FP32 "
      f"({fp32_metrics['size_mb'] / int8_metrics['size_mb']:.2f}x más chico).")
print(f"Drift de cross-entropy: "
      f"{int8_metrics['cross_entropy'] - fp32_metrics['cross_entropy']:+.4f} nats/token.")

modelo         MB   max |torch-onnx|       xent      top-1   respuestas
fp32        26.18          1.431e-05     5.6950    100.0%      100.0%
int8         6.60          4.276e-01     5.6972     93.3%       92.5%

INT8 ocupa 25.21% del FP32 (3.97x más chico).
Drift de cross-entropy: +0.0023 nats/token.


## Self-check

In [12]:
def check_quantized():
    # Una función de comparación que no se reconoce a sí misma como idéntica está rota, y
    # todos los números de la tabla de arriba serían ruido.
    assert top1_agreement(fp32_sess, fp32_sess) == 1.0, \
        "una sesión tiene que coincidir consigo misma el 100% de las veces"
    assert reply_agreement(fp32_sess, fp32_sess) == 1.0, \
        "una sesión tiene que reproducir sus propias respuestas exactamente"

    top1 = top1_agreement(fp32_sess, int8_sess)
    replies = reply_agreement(fp32_sess, int8_sess)
    assert 0.0 <= top1 <= 1.0 and 0.0 <= replies <= 1.0, "la coincidencia es una proporción"
    assert replies <= top1 + 1e-9, \
        "las respuestas enteras no pueden coincidir más seguido que los tokens sueltos"

    # INT8 perturba los pesos; no reemplaza el modelo. Un número muy bajo acá es un bug en
    # la comparación o la sesión equivocada, no daño de la cuantización.
    assert top1 > 0.5, \
        f"una coincidencia top-1 de {top1:.1%} es muy baja para ser solo cuantización"
    drift = cross_entropy(onnx_logits(int8_sess)) - cross_entropy(onnx_logits(fp32_sess))
    assert abs(drift) < 0.5, \
        f"la cross-entropy se movió {drift:+.3f} nats -- eso no es redondeo"

    assert os.path.getsize(int8_path) < 0.6 * os.path.getsize(f"{ONNX_DIR}/tinygpt.onnx"), \
        "el archivo int8 no es realmente más chico -- ¿corrió la cuantización?"

    print(f"coincidencia consigo misma   100.0%  (la medición funciona)")
    print(f"fp32 vs int8     top-1 {top1:.1%} | respuestas {replies:.1%} | "
          f"cross-entropy {drift:+.4f}")
    print("todos los checks pasaron")


check_quantized()

coincidencia consigo misma   100.0%  (la medición funciona)
fp32 vs int8     top-1 93.3% | respuestas 92.5% | cross-entropy +0.0023
todos los checks pasaron


# Consigna V — benchmark

Cuatro formas de producir los mismos tokens: PyTorch, ONNX sin cache, ONNX con cache, e
INT8. Tu `stream_tokens` ya maneja tres de ellas — corrélo greedy, así la comparación es
sobre el runtime y no sobre la suerte del sampleo.

Medí dos cosas por corrida, no una:

- **tiempo de reloj** del loop entero;
- **tiempo de engine** — `ENGINE["seconds"]`, que `run_session` viene acumulando por vos.
  Reseteálo antes de cada corrida y leelo después.

La diferencia entre los dos es Python: tokenizar, cortar, armar el diccionario de entradas,
samplear. Con 6.5M de parámetros esa diferencia no es un error de redondeo, y notarlo es el
punto del ejercicio. También es la razón por la que los servers de verdad no se escriben
como un `for` de Python.

**Reglas para un benchmark que valga la pena reportar:** calentá primero (la primera
llamada a `run` construye los kernels), usá el mismo prompt y la misma cantidad de
tokens en todos lados, y repetí lo suficiente como para ver el ruido. Aclará en qué
máquina corriste.

In [13]:
N_TOKENS, WARMUP, REPEATS = 32, 3, 3


def timed(decode, n_tokens: int = N_TOKENS) -> Tuple[float, float]:
    """
    Calentá, después corré `decode(n_tokens)` y devolvé (segundos de reloj, de engine).

    El tiempo de engine sale de ENGINE["seconds"] -- reseteálo justo antes de medir.
    Quedate con la mejor de REPEATS corridas: el mínimo es la estimación menos ruidosa.
    """
    # TODO: Consigna V
    for _ in range(WARMUP):
        ENGINE["seconds"] = 0.0
        decode(n_tokens)

    runs = []
    for _ in range(REPEATS):
        ENGINE["seconds"] = 0.0
        t0 = time.perf_counter()
        decode(n_tokens)
        wall = time.perf_counter() - t0
        engine = ENGINE["seconds"]
        runs.append((wall, engine))
    return min(runs, key=lambda pair: pair[0])


def torch_decode(n_tokens: int) -> None:
    """Dado: el baseline de PyTorch, greedy, sin cache -- lo que ONNX tiene que ganarle."""
    ids = tokenizer.encode(DEMO_PROMPTS[0])
    for _ in range(n_tokens):
        x = torch.tensor(ids[-config.block_size:], dtype=torch.long)[None, :]
        t0 = time.perf_counter()
        with torch.no_grad():
            logits = model(x)
        ENGINE["seconds"] += time.perf_counter() - t0
        ids.append(int(logits[0, -1].argmax()))


# TODO: Consigna V -- medí estas cuatro e imprimí ms/token, tok/s, el reparto
# engine/python y la aceleración contra torch:
#   torch cpu | onnx fp32 (sin cache) | onnx fp32 (kv cache) | onnx int8 (sin cache)
#
# Para las filas de ONNX llamá a tu propio generador, greedy y sin frenar antes:
#   list(stream_tokens(sess, DEMO_PROMPTS[0], max_new_tokens=n, use_cache=...,
#                      do_sample=False))
decoders = {
    "torch cpu": torch_decode,
    "onnx fp32 (sin cache)": lambda n: list(stream_tokens(
        fp32_sess, DEMO_PROMPTS[0], max_new_tokens=n, use_cache=False, do_sample=False)),
    "onnx fp32 (kv cache)": lambda n: list(stream_tokens(
        kv_sess, DEMO_PROMPTS[0], max_new_tokens=n, use_cache=True, do_sample=False)),
    "onnx int8 (sin cache)": lambda n: list(stream_tokens(
        int8_sess, DEMO_PROMPTS[0], max_new_tokens=n, use_cache=False, do_sample=False)),
}

BENCHMARK_RESULTS = {}
for name, decode in decoders.items():
    wall, engine = timed(decode)
    python_time = max(wall - engine, 0.0)
    BENCHMARK_RESULTS[name] = {
        "wall": wall,
        "engine": engine,
        "python": python_time,
        "ms_per_token": 1000.0 * wall / N_TOKENS,
        "tok_per_s": N_TOKENS / wall,
    }

torch_wall = BENCHMARK_RESULTS["torch cpu"]["wall"]
print(f"{'método':<26} {'ms/token':>10} {'tok/s':>10} {'engine':>10} "
      f"{'python':>10} {'speedup':>10}")
for name, m in BENCHMARK_RESULTS.items():
    engine_pct = 100.0 * m["engine"] / m["wall"] if m["wall"] else 0.0
    python_pct = 100.0 * m["python"] / m["wall"] if m["wall"] else 0.0
    speedup = torch_wall / m["wall"] if m["wall"] else float("inf")
    print(f"{name:<26} {m['ms_per_token']:>10.2f} {m['tok_per_s']:>10.2f} "
          f"{engine_pct:>9.1f}% {python_pct:>9.1f}% {speedup:>9.2f}x")

método                       ms/token      tok/s     engine     python    speedup
torch cpu                        3.98     251.24      86.6%      13.4%      1.00x
onnx fp32 (sin cache)            0.22    4502.98      75.0%      25.0%     17.92x
onnx fp32 (kv cache)             0.36    2816.78      84.9%      15.1%     11.21x
onnx int8 (sin cache)            0.13    7891.88      67.3%      32.7%     31.41x


# Preguntas

Respondelas con los números que acabás de producir. Sin volver a correr nada.

1. Tu modelo INT8 es unas **4x más chico** pero ni cerca de 4x más rápido. Explicá los dos
   números con el mismo hecho sobre esta arquitectura — el TP-2 lo midió, y es la razón por
   la que "parameter-efficient" era una pregunta real allá.
2. El grafo con KV-cache hace estrictamente menos aritmética por token que el que no lo
   tiene. ¿Ganó por el margen que esperabas? Sea cual sea la respuesta, explicala en
   términos de `block_size = 32` y de la proporción de Python en tu propia tabla.
3. La paridad de logits es el test correcto en la Consigna I y el equivocado en la Consigna
   IV. Decí qué mide realmente cada uno de tus tres tests — paridad, cross-entropy,
   coincidencia — y dame un cambio en un modelo que la coincidencia detectaría mientras la
   cross-entropy queda igual.
4. El trazado registra una sola ejecución. Nombrá un cambio en `TinyGPT.forward` que
   exportaría limpio, pasaría un chequeo de paridad de un solo largo, y aun así estaría mal
   en producción — y decí qué línea del self-check de la Consigna II lo atraparía.
5. Exportaste los mismos pesos tres veces. ¿Qué artefacto mandarías realmente a un
   teléfono, y cuál a un server con GPU? Defendé las dos elecciones con un número.


## Respuestas

1. En INT8 el modelo pasó de **26.18 MB a 6.60 MB**, o sea que quedó casi **4 veces más chico**. Esto es esperable porque los pesos pasan de 32 bits a 8 bits. En velocidad, en cambio, la mejora fue menor: ONNX FP32 dio **4503 tok/s** e INT8 **7892 tok/s**, aproximadamente **1.75x**. Esto pasa porque cuantizar reduce mucho el tamaño de los pesos, pero no reduce de la misma manera todo el costo de inferencia. Siguen existiendo otras operaciones y también overhead del runtime y de Python. Además, en este modelo gran parte de los parámetros están en el embedding y en la proyección final al vocabulario, como vimos en el TP-2.

2. En este caso el KV-cache no mejoró el tiempo. Sin cache obtuve **0.22 ms/token (4503 tok/s)** y con cache **0.36 ms/token (2817 tok/s)**. Aunque con cache se hace menos cómputo, `block_size` es solamente 32, por lo que recalcular el contexto tampoco es demasiado costoso. En cambio, aparece el costo de manejar y pasar los tensores del cache en cada paso. También hay una parte del tiempo que corresponde a Python y no al modelo. Con contextos más largos probablemente el beneficio del cache sería más evidente.

3. La paridad de logits sirve para verificar que ONNX esté haciendo prácticamente el mismo cálculo que PyTorch para una misma entrada. La cross-entropy mide qué tan bien modela el lenguaje sobre el conjunto de evaluación. La coincidencia, en cambio, compara directamente las decisiones del modelo FP32 y del cuantizado, ya sea en el próximo token o en la respuesta completa. Por ejemplo, se podrían modificar las probabilidades entre varios tokens sin cambiar demasiado la probabilidad del token correcto. En ese caso la cross-entropy podría mantenerse prácticamente igual, pero cambiar el token elegido como top-1.

4. Un problema posible sería tener dentro de `TinyGPT.forward` una condición de Python que dependa del largo del prefijo o del cache. Al exportar con un solo largo, el tracer podría dejar fija la rama que se ejecutó durante ese ejemplo. El grafo podría pasar la prueba para ese largo y fallar con otros. El self-check de la Consigna II lo detecta porque prueba distintos valores de `past_len` (`0, 3, 8, 20` y `block_size - 1`) y compara tanto los logits como el tamaño del cache resultante.

5. Para un teléfono elegiría el modelo **INT8**, principalmente por tamaño: ocupa **6.60 MB en lugar de 26.18 MB**, y la cross-entropy prácticamente no cambió (**5.6972 vs 5.6950**). Para un servidor con GPU usaría la versión FP32 y, para generación con contextos más grandes, la variante con KV-cache. En este TP el cache no fue más rápido porque `block_size = 32` es muy chico, pero con secuencias más largas evita recalcular todo el contexto para cada nuevo token.



# Consigna VI — un endpoint compatible con OpenAI

El grafo corre. Ahora dale la interfaz que el resto del mundo ya habla.

Todo modelo hosteado — OpenAI, y llama.cpp, vLLM, Ollama y TGI en sus modos de
compatibilidad — responde `POST /v1/chat/completions` con el mismo JSON. Hablá esa forma y
cualquier cliente existente funciona contra tu modelo sin tocarlo. Esa es toda la lección
de esta parte: **el contrato de la API es el límite del producto, no el modelo.**

El request que tenés que aceptar y la respuesta que tenés que devolver:

```json
{ "model": "tinygpt", "messages": [{"role": "user", "content": "continue: To be,"}],
  "max_tokens": 24, "temperature": 0.8 }
```

```json
{ "id": "chatcmpl-...", "object": "chat.completion", "created": 1700000000,
  "model": "tinygpt",
  "choices": [{ "index": 0,
                "message": {"role": "assistant", "content": "..."},
                "finish_reason": "stop" }],
  "usage": {"prompt_tokens": 12, "completion_tokens": 7, "total_tokens": 19} }
```

**Qué implementar**

1. `render_prompt(messages)` — los mensajes de OpenAI al template del TP-2. Un mensaje
   `system` no tiene dónde ir en un template que solo conoce `<|user|>` y `<|assistant|>`:
   decidí qué hacés con él y dejalo dicho en tu informe.
2. `decode_args(req)` — traducir las perillas del request a los argumentos que toma tu
   `stream_tokens` de la Consigna III. `temperature: 0` significa greedy, y hay que
   suprimir `SCAFFOLD_IDS`.
3. `generate_reply(req)` — correr la decodificación. Devolvé el texto **y** por qué
   frenaste, más las cuentas de tokens.
4. `build_response(req)` — armar el JSON de arriba.

Nada de código de decodificación nuevo: la Consigna III ya lo escribió. Esta consigna es la
capa de traducción entre una API pública y tu generador, que es casi todo lo que un stack
de serving hace.

La ruta se registra unas celdas más abajo, una sola vez, cuando ya estén escritas esta
consigna y la VII. Registrar dos veces el mismo path en FastAPI no reemplaza el handler —
gana el primero, en silencio — así que todo el ruteo vive en una única celda dada.

**Tres detalles que son, en realidad, la nota**

- **`finish_reason`** es `"stop"` cuando el modelo emitió `<|end|>` y `"length"` cuando
  llegó a `max_tokens`. Los clientes bifurcan según eso — así sabe quien llama si conviene
  pedir más.
- **`usage`** tiene que contar tokens de verdad: prompt, completion, y la suma.
- **Suprimí el andamiaje.** `SCAFFOLD_IDS` tiene `<|user|>`, `<|assistant|>` y `<|pad|>`.
  Poné esos logits en `-inf` antes de samplear, o van a aparecer *adentro* de las
  respuestas — el mismo bug del que avisaba la Consigna V del TP-2, ahora en producción.

In [14]:
class Message(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str


class ChatRequest(BaseModel):
    model: str = "tinygpt"
    messages: List[Message]
    max_tokens: int = 24
    temperature: float = 0.8
    top_p: float = 1.0            # del contrato de OpenAI -- lo hace la Consigna III
    stream: bool = False          # Consigna VII


app = FastAPI(title="TinyGPT")


def render_prompt(messages: List[Message]) -> str:
    """Mensajes de OpenAI -> el chat template del TP-2, terminando en <|assistant|>."""
    # TODO: Consigna VI
    # TinyGPT no fue entrenado con un rol system. Lo descartamos explícitamente en vez de
    # inventar un formato fuera de distribución; esta decisión se justifica en el informe.
    parts = []
    for msg in messages:
        if msg.role == "system":
            continue
        if msg.role == "user":
            parts.append(f"{USER}{msg.content}")
        elif msg.role == "assistant":
            parts.append(f"{ASSISTANT}{msg.content}{END}")

    prompt = "".join(parts)
    if not prompt.endswith(ASSISTANT):
        prompt += ASSISTANT
    return prompt


def decode_args(req: ChatRequest) -> Dict:
    """
    Los argumentos que tu `stream_tokens` de la Consigna III necesita para este request.

    temperature 0 significa greedy; el andamiaje no puede samplearse nunca.
    """
    # TODO: Consigna VI
    greedy = req.temperature == 0
    return {
        "max_new_tokens": req.max_tokens,
        "use_cache": True,
        "do_sample": not greedy,
        "temperature": 1.0 if greedy else req.temperature,
        "top_p": req.top_p,
        "suppress": SCAFFOLD_IDS,
    }


def generate_reply(req: ChatRequest) -> Tuple[str, str, int, int]:
    """Devuelve (texto, finish_reason, n_tokens_prompt, n_tokens_completion)."""
    # TODO: Consigna VI -- decodificá con stream_tokens; menos tokens que max_tokens
    #                      significa que frenó en <|end|>
    prompt = render_prompt(req.messages)
    produced = list(stream_tokens(kv_sess, prompt, **decode_args(req)))
    text = tokenizer.decode(produced)
    finish_reason = "stop" if len(produced) < req.max_tokens else "length"
    n_prompt = len(tokenizer.encode(prompt))
    n_completion = len(produced)
    return text, finish_reason, n_prompt, n_completion


def build_response(req: ChatRequest) -> Dict:
    """La respuesta sin streaming documentada arriba."""
    # TODO: Consigna VI
    text, finish_reason, n_prompt, n_completion = generate_reply(req)
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": text},
            "finish_reason": finish_reason,
        }],
        "usage": {
            "prompt_tokens": n_prompt,
            "completion_tokens": n_completion,
            "total_tokens": n_prompt + n_completion,
        },
    }

# Consigna VII — streaming

Tu endpoint hace esperar a quien llama hasta el último token antes de mostrarle el primero.
Con 24 tokens en una CPU eso es una pausa; con 500 tokens es una interfaz muerta. Todos los
productos de chat que usaste hacen streaming, y el mecanismo es más viejo y más simple que
el modelo: **server-sent events**, una respuesta HTTP larga que se escribe de a pedazos.

El formato es texto plano. Cada evento es una línea `data:` con un objeto JSON, separados
por una línea en blanco, y el stream termina con un centinela literal:

```
data: {"id":"chatcmpl-abc","object":"chat.completion.chunk","created":1700000000,"model":"tinygpt","choices":[{"index":0,"delta":{"role":"assistant"},"finish_reason":null}]}

data: {"id":"chatcmpl-abc", ... ,"choices":[{"index":0,"delta":{"content":"DUKE"},"finish_reason":null}]}

data: {"id":"chatcmpl-abc", ... ,"choices":[{"index":0,"delta":{},"finish_reason":"stop"}]}

data: [DONE]

```

Tres diferencias con la Consigna VI, y los clientes dependen de las tres:

- `object` es **`chat.completion.chunk`**, y cada choice lleva un **`delta`**, no un
  `message`. El primer delta anuncia el rol, los del medio llevan texto, el último está
  vacío y lleva `finish_reason`.
- El `id` es **el mismo en todos los chunks** de una misma respuesta.
- `usage` no está. No se conoce hasta el final, y OpenAI solo lo manda si se lo pedís con
  `stream_options`. Dejarlo afuera es el default compatible.

**Qué implementar**

`sse_events(req)`, un generador que va entregando esos strings `data: ...\n\n`.
Decodificá iterando tu `stream_tokens` de la Consigna III — un solo loop de decodificación,
dos presentaciones.

**La trampa: no decodifiques los tokens de a uno.** El BPE a nivel de bytes parte
caracteres entre tokens, así que `tokenizer.decode([one_id])` puede devolverte medio
carácter. Decodificá todo lo producido hasta ahora y emití solo la parte que todavía no
mandaste:

```python
text = tokenizer.decode(produced)
delta, sent = text[len(sent):], text
```

In [15]:
def sse_chunk(chunk_id: str, model: str, delta: Dict,
              finish_reason: Optional[str]) -> str:
    """Un evento `data: {...}`, línea en blanco incluida. Dado -- la forma es el contrato."""
    payload = {
        "id": chunk_id,
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "delta": delta, "finish_reason": finish_reason}],
    }
    return f"data: {json.dumps(payload)}\n\n"


def sse_events(req: ChatRequest):
    """
    Va entregando el stream de eventos de un request.

    Orden: un delta con el rol, después deltas de contenido, después un delta vacío que
    lleva finish_reason, y al final `data: [DONE]`.
    """
    # TODO: Consigna VII
    chunk_id = f"chatcmpl-{uuid.uuid4().hex}"
    yield sse_chunk(chunk_id, req.model, {"role": "assistant"}, None)

    prompt = render_prompt(req.messages)
    produced = []
    sent = ""

    for token_id in stream_tokens(kv_sess, prompt, **decode_args(req)):
        produced.append(token_id)
        text = tokenizer.decode(produced)
        delta = text[len(sent):]
        sent = text
        if delta:
            yield sse_chunk(chunk_id, req.model, {"content": delta}, None)

    finish_reason = "stop" if len(produced) < req.max_tokens else "length"
    yield sse_chunk(chunk_id, req.model, {}, finish_reason)
    yield "data: [DONE]\n\n" 

## Levantalo

La ruta se registra acá, una sola vez, y despacha según `req.stream` a las dos cosas que
escribiste. Después `uvicorn` corre en un thread de fondo para que el notebook siga usable,
atado a `127.0.0.1`, así que nada fuera de esta máquina lo alcanza.

Los dos self-checks llaman al endpoint por HTTP de verdad y leen el contrato de vuelta.

In [16]:
@app.post("/v1/chat/completions")
def chat_completions(req: ChatRequest):
    """La única ruta. La Consigna VI responde de una, la VII responde de a pedazos."""
    if req.stream:
        return StreamingResponse(sse_events(req), media_type="text/event-stream")
    return build_response(req)


PORT = 8000
_config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="error")
server = uvicorn.Server(_config)
threading.Thread(target=server.run, daemon=True).start()

for _ in range(200):                      # toma el socket en bastante menos de un segundo
    if server.started:
        break
    time.sleep(0.05)
print(f"sirviendo en http://127.0.0.1:{PORT}  (started={server.started})")


def check_api():
    url = f"http://127.0.0.1:{PORT}/v1/chat/completions"
    body = {"model": "tinygpt", "temperature": 0.0, "max_tokens": 16,
            "messages": [{"role": "user", "content": "who said: Now is the winter"}]}
    r = requests.post(url, json=body, timeout=60)
    assert r.status_code == 200, f"{r.status_code}: {r.text[:200]}"
    data = r.json()

    for key in ("id", "object", "created", "model", "choices", "usage"):
        assert key in data, f"a la respuesta le falta {key!r}"
    assert data["object"] == "chat.completion", data["object"]
    choice = data["choices"][0]
    assert choice["message"]["role"] == "assistant", choice["message"]
    assert choice["finish_reason"] in ("stop", "length"), choice["finish_reason"]

    usage = data["usage"]
    assert usage["prompt_tokens"] + usage["completion_tokens"] == usage["total_tokens"], usage
    assert usage["completion_tokens"] > 0, "no se generó nada"

    content = choice["message"]["content"]
    for tok in (USER, ASSISTANT, PAD):
        assert tok not in content, f"{tok} se filtró en la respuesta -- suprimí SCAFFOLD_IDS"

    # max_tokens es una promesa, no una sugerencia.
    short = requests.post(url, json={**body, "max_tokens": 2}, timeout=60).json()
    assert short["usage"]["completion_tokens"] <= 2, short["usage"]
    if short["choices"][0]["finish_reason"] == "length":
        assert short["usage"]["completion_tokens"] == 2, short["usage"]

    # La temperatura tiene que llegar de verdad al sampler. Greedy dos veces da lo mismo;
    # samplear caliente varias veces no puede dar siempre lo mismo.
    greedy = {requests.post(url, json={**body, "temperature": 0.0}, timeout=60)
              .json()["choices"][0]["message"]["content"] for _ in range(2)}
    assert len(greedy) == 1, "temperature=0 tiene que ser determinista"
    hot = {requests.post(url, json={**body, "temperature": 2.0}, timeout=60)
           .json()["choices"][0]["message"]["content"] for _ in range(6)}
    assert len(hot) > 1, "temperature=2.0 dio una sola respuesta -- ¿llega a sample_next?"

    # Un segundo turno tiene que llegar al modelo: una API de chat que se olvida la
    # conversación está rota, no es una decisión de diseño.
    multi = requests.post(url, json={**body, "messages": [
        {"role": "user", "content": "who said: Now is the winter"},
        {"role": "assistant", "content": "GLOUCESTER"},
        {"role": "user", "content": "continue: To be, or not"}]}, timeout=60).json()
    assert multi["usage"]["prompt_tokens"] > usage["prompt_tokens"], \
        "una conversación de varios turnos no produjo más tokens de prompt que un solo mensaje"

    # El rol system es una decisión de diseño, no una regla -- esto informa, no verifica.
    sys_usage = requests.post(url, json={**body, "messages": [
        {"role": "system", "content": "You are a Shakespearean actor."},
        {"role": "user", "content": "who said: Now is the winter"}]},
        timeout=60).json()["usage"]
    reached = sys_usage["prompt_tokens"] > usage["prompt_tokens"]

    print(f"contenido    {content!r}")
    print(f"finish       {choice['finish_reason']}")
    print(f"usage        {usage['prompt_tokens']}+{usage['completion_tokens']}"
          f"={usage['total_tokens']}")
    print(f"rol system   {'llega al modelo' if reached else 'se descarta'}"
          f"  ({usage['prompt_tokens']} -> {sys_usage['prompt_tokens']} tokens de prompt)"
          f"  -- defendé tu decisión en las preguntas")
    print("todos los checks pasaron")


def check_stream():
    url = f"http://127.0.0.1:{PORT}/v1/chat/completions"
    body = {"model": "tinygpt", "temperature": 0.0, "max_tokens": 16, "stream": True,
            "messages": [{"role": "user", "content": "continue: To be, or not to be,"}]}
    r = requests.post(url, json=body, stream=True, timeout=60)
    assert r.status_code == 200, f"{r.status_code}: {r.text[:200]}"
    assert "text/event-stream" in r.headers.get("content-type", ""), r.headers

    events = [l[len("data: "):] for l in r.iter_lines(decode_unicode=True)
              if l and l.startswith("data: ")]
    assert events, "no llegó ningún evento data:"
    assert events[-1] == "[DONE]", f"el stream termina con {events[-1]!r}, no con [DONE]"

    chunks = [json.loads(e) for e in events[:-1]]
    assert all(c["object"] == "chat.completion.chunk" for c in chunks), \
        "mientras se streamea, object tiene que ser chat.completion.chunk"
    assert len({c["id"] for c in chunks}) == 1, "todos los chunks necesitan el mismo id"
    assert chunks[0]["choices"][0]["delta"].get("role") == "assistant", \
        "el primer delta anuncia el rol"
    assert all(c["choices"][0]["finish_reason"] is None for c in chunks[:-1]), \
        "solo el último chunk lleva finish_reason"
    assert chunks[-1]["choices"][0]["finish_reason"] in ("stop", "length"), chunks[-1]

    text = "".join(c["choices"][0]["delta"].get("content", "") for c in chunks)
    assert text, "no se streameó ningún contenido"
    for tok in (USER, ASSISTANT, PAD):
        assert tok not in text, f"{tok} se filtró en el stream"
    assert "�" not in text, \
        "carácter de reemplazo -- estás decodificando los tokens de a uno"

    print(f"chunks       {len(chunks)} + [DONE]")
    print(f"streameado   {text!r}")
    print(f"finish       {chunks[-1]['choices'][0]['finish_reason']}")
    print("todos los checks pasaron")


check_api()
print()
check_stream()

sirviendo en http://127.0.0.1:8000  (started=True)
contenido    ", I'll the"
finish       stop
usage        9+4=13
rol system   se descarta  (9 -> 9 tokens de prompt)  -- defendé tu decisión en las preguntas
todos los checks pasaron

chunks       5 + [DONE]
streameado   '\nAnd the'
finish       stop
todos los checks pasaron


# Consigna VIII — llamalo con el cliente de OpenAI

Los self-checks hablaron HTTP crudo. Esta es la parte que justifica todo el laburo: el
**cliente oficial `openai`**, la misma librería que le habla a GPT-5, apuntada a tu laptop.

No se modifica nada del cliente. Manda su request de siempre, parsea la respuesta a su
objeto `ChatCompletion` de siempre, y nunca se entera de que detrás de la URL hay un modelo
de 6.5M de parámetros corriendo en una CPU.

Hacelo de las dos formas:

- **completo** — unos cuantos mensajes, incluyendo uno que frene en `<|end|>` y otro que se
  quede sin `max_tokens`, leyendo los dos valores de `finish_reason`;
- **streaming** — con `stream=True`, imprimiendo cada `chunk.choices[0].delta.content` a
  medida que llega. El cliente te entrega objetos `ChatCompletionChunk` parseados del SSE
  que escribiste en la Consigna VII.

Si el texto streameado y el completo pueden diferir para el mismo prompt, explicá por qué
en tu informe.

In [18]:
from openai import OpenAI

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key="not-needed")

# TODO: Consigna VIII
#   a) mandá al menos tres mensajes e imprimí respuesta, finish_reason y usage de cada uno;
#      incluí uno que frene en <|end|> y otro que se quede sin max_tokens.
#   b) mandá uno con stream=True e imprimí los deltas a medida que llegan.
examples = [
    {
        "messages": [{"role": "user", "content": "who said: Now is the winter of our discontent"}],
        "max_tokens": 24,
        "temperature": 0.0,
    },
    {
        "messages": [{"role": "user", "content": "continue: To be, or not to be,"}],
        "max_tokens": 4,  # deliberadamente corto: fuerza finish_reason='length' si no aparece END
        "temperature": 0.0,
    },
    {
        "messages": [
            {"role": "user", "content": "who said: Now is the winter"},
            {"role": "assistant", "content": "GLOUCESTER"},
            {"role": "user", "content": "continue: To be, or not"},
        ],
        "max_tokens": 24,
        "temperature": 0.8,
    },
]

for i, ex in enumerate(examples, 1):
    r = client.chat.completions.create(model="tinygpt", **ex)
    print(f"\n--- ejemplo {i} ---")
    print("respuesta:", repr(r.choices[0].message.content))
    print("finish_reason:", r.choices[0].finish_reason)
    print("usage:", r.usage)

print("\n--- streaming ---")
stream = client.chat.completions.create(
    model="tinygpt",
    messages=[{"role": "user", "content": "continue: To be, or not to be,"}],
    max_tokens=24,
    temperature=0.0,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()


--- ejemplo 1 ---
respuesta: 'QUEEN ELIZABETH'
finish_reason: stop
usage: CompletionUsage(completion_tokens=7, prompt_tokens=12, total_tokens=19, completion_tokens_details=None, prompt_tokens_details=None)

--- ejemplo 2 ---
respuesta: '\nAnd the'
finish_reason: stop
usage: CompletionUsage(completion_tokens=3, prompt_tokens=12, total_tokens=15, completion_tokens_details=None, prompt_tokens_details=None)

--- ejemplo 3 ---
respuesta: ''
finish_reason: stop
usage: CompletionUsage(completion_tokens=0, prompt_tokens=24, total_tokens=24, completion_tokens_details=None, prompt_tokens_details=None)

--- streaming ---

And the


# Preguntas

1. `finish_reason` distingue `"stop"` de `"length"`. ¿Qué hace distinto quien llama en cada
   caso, y qué se rompe en una interfaz de chat si devolvés siempre `"stop"`?
2. `usage` reporta cuentas de tokens para un modelo que nadie está facturando. Nombrá dos
   cosas además de la facturación para las que quien llama usa esos números. ¿Por qué no
   está mientras se hace streaming, y qué le cuesta eso a quien llama?
3. Tu `render_prompt` tuvo que decidir qué significa un mensaje `system` para un modelo que
   nunca se fine-tuneó con uno. Decí qué elegiste y qué costarían las alternativas. ¿Qué
   haría falta para soportarlo en serio?
4. El streaming devuelve los mismos tokens en el mismo orden y no cambia ninguna métrica de
   la Consigna V — el tiempo total es idéntico. Nombrá qué sí mejora, en un término que un
   usuario reconocería, y explicá por qué importa más con 500 tokens que con 24.
5. El código de streaming tuvo que decodificar toda la respuesta en cada paso en vez de
   token por token. Explicá la falla usando BPE a nivel de bytes, y conectala con por qué
   la tarea `reverse` del TP-2 era imposible.
6. Este server atiende un request por vez, en Python, sin batching ni autenticación. Elegí
   **una** sola cosa que agregarías primero para un despliegue real y justificala contra
   tus números de la Consigna V.
7. En este notebook nunca cambiaste el modelo, solo cómo está representado y cómo se lo
   alcanza. Argumentá a favor o en contra: "la parte difícil del despliegue no es el
   modelo".


## Respuestas

1. "stop" indica que el modelo terminó porque generó <|end|>, mientras que "length" indica que llegó al límite de max_tokens. Esa diferencia le permite al cliente saber si la respuesta terminó realmente o si quedó cortada. Si siempre se devolviera "stop", una respuesta truncada podría mostrarse como si estuviera completa.

2. usage también sirve para controlar cuánto contexto se está usando y para medir el consumo de recursos del servicio. Por ejemplo, se puede usar para ver cuántos tokens procesa cada request o para estimar capacidad y rendimiento. En streaming no se conoce la cantidad final de tokens hasta que termina la generación, por eso no aparece en cada chunk. Si el cliente necesita ese dato durante el stream, tendría que calcularlo por su cuenta o recibirlo al final.

3. Decidí ignorar los mensajes system porque este modelo no fue entrenado con ese rol. Podría haberlo agregado como parte del texto del usuario, pero sería un formato distinto al usado durante el entrenamiento. También se podría crear un token <|system|>, pero no tendría sentido sin entrenar su embedding. Para soportar ese rol correctamente habría que modificar el template y fine-tunear el modelo con ejemplos que incluyan mensajes system.

4. El streaming mejora principalmente la experiencia del usuario porque reduce el tiempo hasta que aparece el primer texto. El tiempo total de generación puede ser el mismo, pero la respuesta empieza a mostrarse mientras se sigue generando. Con 24 tokens la diferencia puede ser pequeña, pero con 500 tokens esperar hasta el final haría que la aplicación parezca trabada.

5. Como el tokenizer usa BPE a nivel de bytes, un carácter puede quedar dividido entre más de un token. Por eso decodificar cada token por separado puede generar caracteres incompletos o �. La solución es decodificar todos los tokens generados hasta ese momento y enviar solamente la parte nueva. Esto también explica el problema de reverse del TP-2: invertir tokens no es necesariamente lo mismo que invertir caracteres, porque los límites de los tokens no coinciden siempre con los límites del texto.

6. Lo primero que agregaría sería batching de requests. En este servidor cada request se procesa por separado, por lo que hay overhead que se repite para cada uno. Agrupar varios requests permitiría aprovechar mejor el engine y mejorar el throughput. Esto se relaciona con lo observado en la Consigna V, donde una parte del tiempo medido no corresponde solamente al cálculo del modelo.

7. Estoy de acuerdo con la frase. En este TP el modelo fue siempre el mismo, pero para poder usarlo fuera de PyTorch hubo que exportarlo, manejar el KV-cache, cuantizarlo, validar que siguiera comportándose correctamente, medir rendimiento y después implementar una API y streaming. El modelo es una parte importante, pero llevarlo a un entorno de uso real requiere resolver varios problemas adicionales.

## Bajalo

El thread es daemon, así que muere con el kernel — pero dejar un socket tomado mientras
seguís trabajando es la forma más rápida de que la próxima corrida te salude con *address
already in use*.

In [19]:
server.should_exit = True
time.sleep(1.0)
print("server detenido")

try:                                  # el visor de Netron, si levantaste uno
    import netron
    netron.stop()
except Exception:
    pass

print("\nartefactos en ./onnx:")
for f in sorted(os.listdir(ONNX_DIR)):
    print(f"  {f:<24} {os.path.getsize(os.path.join(ONNX_DIR, f)) / 1e6:>7.1f} MB")

server detenido

artefactos en ./onnx:
  tinygpt.int8.onnx            6.6 MB
  tinygpt.kv.onnx             26.2 MB
  tinygpt.onnx                26.2 MB
  tinygpt.onnx.data           26.2 MB


# Conclusiones

- La exportación a ONNX no se valida porque el grafo “corra”, sino por paridad en múltiples formas de entrada; con KV-cache, el estado pasa a formar parte explícita del contrato del modelo.

- La cuantización INT8 reduce fuertemente el tamaño del artefacto y puede mantener casi intacta la calidad promedio, aunque la coincidencia exacta de respuestas sea más sensible y la mejora de velocidad no tenga por qué seguir el mismo factor que la compresión.

- El serving agrega problemas que no pertenecen al `forward`: sampling, cache, métricas de latencia, contrato compatible con OpenAI, `usage`, `finish_reason` y streaming seguro para un tokenizer BPE.


# ¡Felicitaciones! 🎉

A lo largo de tres notebooks preentrenaste un GPT, lo hiciste sparse con una Mixture of
Experts, le enseñaste a seguir instrucciones con fine-tuning completo y con LoRA, lo
exportaste a un grafo portable, lo cuantizaste, y lo serviste detrás de la misma API que
habla el resto de la industria.

Preentrenamiento, alineación y despliegue — el pipeline entero, a una escala que podés leer
de punta a punta.